# FLN Fast Scraper — All Levels
Concurrent downloads from Wikimedia Commons API + DuckDuckGo.
No expensive per-image B&W filter — let OpenCV handle conversion later.
Uses the prompt CSV (`fln_level_prompts.csv`) to know what to search for.

In [ ]:
# MOUNT GOOGLE DRIVE (for reliable persistent storage)
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE = Path("/content/drive/MyDrive/FLN_Output")
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print(f"\nGoogle Drive mounted. Output will save to: {DRIVE_BASE}")


In [ ]:
import subprocess, sys, json
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for pkg in ["duckduckgo_search", "requests", "Pillow", "imagehash", "numpy"]:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"  OK {pkg}")
    except ImportError:
        print(f"  Installing {pkg}...")
        install(pkg)
print("Done.")

In [ ]:
from pathlib import Path
import os, io, time, requests, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image

BASE = Path(input(f"Output to Drive [default: {DRIVE_BASE}]: ").strip() or str(DRIVE_BASE))
IMAGES_PER_SUBLEVEL = int(input("Images per sub-level [default: 12]: ").strip() or "12")
LEVELS_IN = input("Level numbers to scrape (comma-sep, or 'all') [default: all]: ").strip()
MIN_SIZE = 10000
MAX_SIZE = 5000000
VALID_EXT = {".jpg", ".jpeg", ".png", ".webp"}
SUBLABELS = ["Mastery", "Easier_Remediation", "Further_Remediation"]
IMAGE_STYLE = "minimalist black white kindergarten worksheet line art clean outline flat 2d monochrome clip art"  # appended to all search queries
print(f"\nOutput: {BASE}")
print(f"Target: {IMAGES_PER_SUBLEVEL} per sub-level")


In [ ]:
# UPLOAD the prompt CSV file (fln_level_prompts.csv)
from google.colab import files
print("Please upload fln_level_prompts.csv:")
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0] if uploaded else "/content/fln_level_prompts.csv"
print(f"Using: {CSV_PATH}")

import csv
LEVEL_DATA = {}
with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    for row in reader:
        lvl = int(row["level"])
        if lvl not in LEVEL_DATA:
            LEVEL_DATA[lvl] = {
                "name": row["name"],
                "class_range": row["class_range"],
                "nipun_strand": row["nipun_strand"],
                "queries": row["image_search_prompts"].split("; ") if row.get("image_search_prompts") else [],
            }

if LEVELS_IN.lower() == "all":
    LEVELS = sorted(LEVEL_DATA.keys())
else:
    LEVELS = [int(x.strip()) for x in LEVELS_IN.split(",") if x.strip()]

print(f"Loaded {len(LEVEL_DATA)} levels from CSV")
print(f"Scraping levels: {LEVELS}")
for l in LEVELS:
    d = LEVEL_DATA.get(l, {})
    print(f"  {l:2d}: {d.get('name', '?')} ({d.get('class_range', '?')}) — {len(d.get('queries', []))} queries")


In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 FLN-Project/1.0"})

def is_bw(img):
    if img.mode == "1" or img.mode == "L":
        return True
    if img.mode != "RGB":
        return False
    arr = __import__("numpy", fromlist=[""]).array(img)
    r, g, b = arr[:,:,0].astype(float), arr[:,:,1].astype(float), arr[:,:,2].astype(float)
    sat = (abs(r - g) + abs(g - b) + abs(b - r)) / 3
    return (sat > 30).mean() < 0.05

def safe_name(url, idx):
    ext = os.path.splitext(url.split("?")[0])[1].lower()
    if ext not in VALID_EXT:
        ext = ".jpg"
    return f"{idx:03d}{ext}"

def download_one(url, dest, timeout=12):
    try:
        r = session.get(url, timeout=timeout)
        r.raise_for_status()
        c = r.content
        if len(c) < MIN_SIZE or len(c) > MAX_SIZE:
            return False
        img = Image.open(io.BytesIO(c))
        if img.width < 200 or img.height < 200:
            return False
        if not is_bw(img):
            return False
        img.save(dest)
        return True
    except Exception:
        return False

def download_batch(urls, dest_dir):
    existing = {p.stem for p in Path(dest_dir).iterdir()} if os.path.exists(dest_dir) else set()
    results = []
    idx = len(existing) + 1
    with ThreadPoolExecutor(max_workers=15) as pool:
        fut_map = {}
        for url in urls:
            if not url:
                continue
            fname = safe_name(url, idx + len(fut_map))
            dst = Path(dest_dir) / fname
            if dst.exists() or fname.rsplit(".", 1)[0] in existing:
                continue
            fut = pool.submit(download_one, url, dst)
            fut_map[fut] = (url, dst)
        for fut in as_completed(fut_map):
            url, dst = fut_map[fut]
            if fut.result():
                results.append(dst)
    return results

In [ ]:
# SCRAPER 1: DuckDuckGo (returns list of image URLs)
from duckduckgo_search import DDGS

def scrape_ddg_urls(search_term, max_results=15):
    urls = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.images(search_term, region="wt-wt", safesearch="off", max_results=max_results):
                url = r.get("image", "")
                if url:
                    urls.append(url)
    except Exception:
        pass
    return urls

In [ ]:
# SCRAPER 2: Wikimedia Commons API (no rate limit, returns list of image URLs)
def scrape_wiki_urls(search_term, max_results=15):
    urls = []
    try:
        params = {
            "action": "query", "list": "search",
            "srsearch": f"{search_term} filetype:png OR filetype:jpg",
            "srnamespace": "6", "format": "json", "srlimit": max_results * 2
        }
        r = requests.get("https://commons.wikimedia.org/w/api.php", params=params, timeout=15)
        data = r.json()
        for result in data.get("query", {}).get("search", []):
            title = result.get("title", "")
            if not title.startswith("File:"):
                continue
            img_params = {
                "action": "query", "titles": title,
                "prop": "imageinfo", "iiprop": "url", "format": "json",
            }
            ir = requests.get("https://commons.wikimedia.org/w/api.php", params=img_params, timeout=10)
            idata = ir.json()
            for pid, pinfo in idata.get("query", {}).get("pages", {}).items():
                if pid == "-1":
                    continue
                for ii in pinfo.get("imageinfo", []):
                    url = ii.get("url", "")
                    if url:
                        urls.append(url)
    except Exception:
        pass
    return urls

In [ ]:
# SCRAPER 3: Rapid multi-source gathering for one sub-level
def fill_sublevel(sub_dir, queries, target_count):
    os.makedirs(sub_dir, exist_ok=True)
    existing = len([f for f in os.listdir(sub_dir) if f.lower().endswith(tuple(VALID_EXT))])
    if existing >= target_count:
        return existing
    needed = target_count - existing
    print(f"    Need {needed} more...")

    # Phase 1: Fire all DDG + Wiki searches in parallel for all queries
    all_urls = []
    with ThreadPoolExecutor(max_workers=20) as pool:
        ddg_futs = [pool.submit(scrape_ddg_urls, f"{q} {IMAGE_STYLE}", max(needed, 8)) for q in queries]
        wiki_futs = [pool.submit(scrape_wiki_urls, f"{q} {IMAGE_STYLE}", max(needed, 8)) for q in queries]
        for fut in as_completed(ddg_futs + wiki_futs):
            urls = fut.result()
            all_urls.extend(urls)

    # Deduplicate URLs
    seen = set()
    unique_urls = []
    for u in all_urls:
        if u not in seen:
            seen.add(u)
            unique_urls.append(u)
    print(f"    Got {len(unique_urls)} unique URLs from all sources")

    # Phase 2: Download all URLs concurrently in batches
    batch_size = needed * 3
    downloaded = download_batch(unique_urls[:batch_size], sub_dir)
    print(f"    Downloaded {len(downloaded)} images in this batch")

    final = len([f for f in os.listdir(sub_dir) if f.lower().endswith(tuple(VALID_EXT))])
    if final < target_count and len(unique_urls) > batch_size:
        more = download_batch(unique_urls[batch_size:batch_size*2], sub_dir)
        print(f"    Extra batch: {len(more)} more images")
        final = len([f for f in os.listdir(sub_dir) if f.lower().endswith(tuple(VALID_EXT))])
    return final

In [ ]:
# MAIN: Scrape all selected levels
os.makedirs(BASE, exist_ok=True)
grand_total = 0
time_start = time.time()

print("=" * 60)
print(f"  FLN FAST SCRAPER — {len(LEVELS)} levels")
print(f"  {IMAGES_PER_SUBLEVEL} images per sub-level × 3 = {IMAGES_PER_SUBLEVEL*3} per level")
print(f"  Sources: Wikimedia Commons + DuckDuckGo (concurrent)")
print("=" * 60)

for lvl in LEVELS:
    info = LEVEL_DATA.get(lvl)
    if not info:
        print(f"\nLevel {lvl}: no data in CSV, skipping")
        continue
    name = info["name"]
    safe = name.replace(" ", "_").replace("+", "_").replace(",", "").replace("-", "_")
    lvl_dir = BASE / "pinterest_images" / f"Level_{lvl:02d}_{safe}"
    queries = info["queries"]

    print(f"\nLevel {lvl}: {name}")
    print("-" * 50)
    lvl_total = 0

    for sub_num in range(3):
        sub_label = SUBLABELS[sub_num]
        sub_dir = lvl_dir / f"{lvl}.{sub_num}_{sub_label}"
        final = fill_sublevel(sub_dir, queries, IMAGES_PER_SUBLEVEL)
        print(f"    -> {lvl}.{sub_num} {sub_label}: {final} images")
        lvl_total += final

    print(f"  Level total: {lvl_total} images")
    grand_total += lvl_total

elapsed = time.time() - time_start
print(f"\n{'=' * 60}")
print(f"  DONE: {grand_total} images across {len(LEVELS)} levels")
print(f"  Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"  Output: {BASE}/pinterest_images/")

In [ ]:
# DEDUPLICATE within and across sub-levels
import imagehash

def dedup_dir(dir_path):
    hashes, removed = {}, 0
    for fname in sorted(os.listdir(dir_path)):
        fpath = os.path.join(dir_path, fname)
        if not fname.lower().endswith(tuple(VALID_EXT)):
            continue
        try:
            h = imagehash.average_hash(Image.open(fpath))
            if h in hashes:
                os.remove(fpath)
                removed += 1
            else:
                hashes[h] = fpath
        except:
            pass
    return removed

def dedup_cross(parent):
    seen, removed = set(), 0
    for subdir in sorted(parent.iterdir()):
        if not subdir.is_dir():
            continue
        for fname in sorted(os.listdir(subdir)):
            fpath = subdir / fname
            if not fname.lower().endswith(tuple(VALID_EXT)):
                continue
            try:
                h = imagehash.average_hash(Image.open(fpath))
                if h in seen:
                    os.remove(fpath)
                    removed += 1
                else:
                    seen.add(h)
            except:
                pass
    return removed

print("=== DEDUPLICATION ===\n")
for lvl in LEVELS:
    info = LEVEL_DATA[lvl]
    safe = info["name"].replace(" ", "_").replace("+", "_").replace(",", "").replace("-", "_")
    lvl_dir = BASE / "pinterest_images" / f"Level_{lvl:02d}_{safe}"
    if not lvl_dir.exists():
        continue
    for sub_num in range(3):
        sub_dir = lvl_dir / f"{lvl}.{sub_num}_{SUBLABELS[sub_num]}"
        if sub_dir.exists():
            r = dedup_dir(sub_dir)
            c = len([f for f in os.listdir(sub_dir) if f.lower().endswith(tuple(VALID_EXT))])
            if r:
                print(f"  Level {lvl}.{sub_num}: removed {r} dupes, {c} remain")
    r2 = dedup_cross(lvl_dir)
    if r2:
        print(f"  Level {lvl}: removed {r2} cross-sub-level dupes")
print("\nDone.")

In [ ]:
# GENERATE HTML TABLE (Q No. | Illustration | Question)
from collections import OrderedDict

def generate_table():
    rows = []
    pi_dir = BASE / "pinterest_images"
    if not pi_dir.exists():
        print("No pinterest_images directory.")
        return rows
    for ld in sorted(pi_dir.iterdir()):
        if not ld.is_dir():
            continue
        for sd in sorted(ld.iterdir()):
            if not sd.is_dir():
                continue
            parts = sd.name.split("_", 1)
            sub_code = parts[0]
            imgs = sorted([f for f in sd.iterdir() if f.suffix.lower() in VALID_EXT])
            for i, ip in enumerate(imgs, 1):
                rows.append({"q": i, "img": str(ip.relative_to(BASE)), "code": sub_code})
    return rows

def export_html(rows):
    if not rows:
        print("No rows")
        return
    groups = OrderedDict()
    for r in rows:
        groups.setdefault(r["code"], []).append(r)
    h = ['<html><head><meta charset="utf-8"><style>',
         'body{font-family:Cambria,serif;font-size:11pt;margin:20px}',
         'h2{color:#1a5276}', 'table{border-collapse:collapse;width:100%;margin-bottom:30px}',
         'th,td{border:1px solid #999;padding:6px 10px}',
         'th{background:#d4e6f1}', 'img{max-width:120px;max-height:120px}',
         '</style></head><body>']
    h.append('<h1>FLN Worksheets — All Levels</h1>')
    for code, group in groups.items():
        h.append(f'<h2>Level {code}</h2><table><tr><th>Q</th><th>Illustration</th><th>Question</th></tr>')
        for r in group:
            h.append(f'<tr><td>{r["q"]}</td><td><img src="file://{BASE / r["img"]}"></td><td></td></tr>')
        h.append('</table>')
    h.append('</body></html>')
    out = BASE / "fln_tables.html"
    with open(out, "w") as f:
        f.write('\n'.join(h))
    print(f"Exported {len(rows)} rows -> {out}")

print("\nGenerating table...")
rows = generate_table()
export_html(rows)

In [ ]:
# VERIFY: Everything saved directly to Google Drive
print("=" * 60)
print(f"  ALL FILES SAVED TO GOOGLE DRIVE")
print("=" * 60)
print(f"  Location: {BASE}")
print()
pi_dir = BASE / "pinterest_images"
if pi_dir.exists():
    total = 0
    for d in sorted(pi_dir.iterdir()):
        if not d.is_dir():
            continue
        imgs = sum(1 for f in d.rglob('*') if f.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})
        print(f"  {d.name}: {imgs} images")
        total += imgs
    print(f"\n  TOTAL: {total} images across all levels")
    print(f"  HTML table: {BASE / 'fln_tables.html'}")
    print()
    print("  ✅ Open Google Drive → FLN_Output/ to access everything")
    print("  ✅ No zip download needed — files persist in Drive")
else:
    print("No output found yet.")
